In [9]:

from pathlib import Path
import sys
from src.utils import load_file
from src.practice2.document_paser import parse_manual
from src.practice2.text_chunker import chunk_documents
from src.practice2.embeddings import vectorize_chunks
from src.practice2.retriever import retrieve
from src.practice2.generator import generate_answer

PROJECT_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))
manual_path = PROJECT_DIR / "data" / "manual.pdf"

chunk_size = 1000
chunk_overlap = 150
top_k = 4

mineru_mode = "flash"
embed_model = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

gen_model = "gpt-4.1-mini"
api_key = load_file(".env")

In [6]:
docs = parse_manual(manual_path, mode=mineru_mode)


[1/5] Parsing with MinerU (mode=flash)...
- parsed documents/pages: 1
- saved: c:\Users\jia\Downloads\llm\LLM_Practice\src\output\parsed.md


In [7]:
chunks = chunk_documents(
    docs,
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

print("Number of chunks:", len(chunks))
print(chunks[0].page_content)


[2/5] Chunking...
- chunks: 40
- saved: c:\Users\jia\Downloads\llm\LLM_Practice\src\output\chunks.jsonl
Number of chunks: 40
## 사용 설명서

## Music Studio 7 (HW-LS7\*H) Music Studio 5 (HW-LS5\*H)

보다 완벽한 서비스를 받으시려면 구입하신 제품을 www.samsung.com에서등록하시기 바랍니다.사용 설명서 상의 그림은 설명을 위해 그려진 것으로 실제제품 외관과 다를 수 있습니다.

제품 디자인 및 제품 규격은 제품 성능 향상을 위해 사전 예고 없이 변경될 수있습니다

## 목차


In [8]:
encoder, vectors = vectorize_chunks(
    chunks,
    embed_model,
)

print("Embedding shape:", vectors.shape)


[3/5] Vectorizing with sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2...


c:\Users\jia\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jia\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\jia\AppData\Local\Python\pythoncore-3.14-64\Lib\s

- embedding shape: (40, 384)
- saved: c:\Users\jia\Downloads\llm\LLM_Practice\src\output\embeddings.npy
Embedding shape: (40, 384)


In [10]:
query = "What is camera calibration?"

retrieved = retrieve(
    query,
    encoder,
    vectors,
    chunks,
    top_k=top_k,
)


[4/5] Retrieving relevant chunks...
#1 score=0.281, page=?: rowspan=1 colspan=1><img src="images/507bfc587a39ae9eff676d8452d8d9a0c28f2ff44613644011ab00023423684a.jpg"/></td><td rowspan=1 colspan=1>주의,...
#2 score=0.269, page=?: <table><tr><td colspan="1" rowspan="1">음향모드</td><td colspan="1" rowspan="1">•음악:음악 콘텐츠를 위해 더 자연스럽고 공간을 가득채우는 듯한 사운드를 제공합니다.표준:오리지널 사운드로 감상할 ...
#3 score=0.257, page=?: ·가능하면 코어리스 HDMI케이블을 사용하는 것이 좋습니다.코어형 HDMI케이블을 사용하는 경우에는 직경이 14mm미만인 것을 사용하세요.  ·HDMI케이블이 ARC를 지원하지 않는 경우에는 이 기능을 사용할 수없습니다.  방송 신호가 Dolby일 때...
#4 score=0.247, page=?: 재생</td><td colspan="1" rowspan="1">감상 중이었던 음악을 바로 재생 할 수 있습니다.("즉시 재생 기능 사용하기"색션을 참조하세요.)</td></tr><tr><td>음향 세부항목 설정</td><td>Voice 음향 강화는 동...


In [ ]:
answer = generate_answer(
    query,
    retrieved,
    gen_model,
    api_key,
)

print(answer)